# Módulo 4 · Notebook 3 — RAG + Memoria

Un asistente real necesita **dos fuentes de contexto**:

| Capa | Qué recuerda | Ejemplo |
|------|--------------|--------|
| **Memoria** | Turnos del chat (sesión) | "Me llamo Ana" |
| **RAG** | Documentos indexados | "¿Qué dice el paper sobre FinGPT?" |

Este notebook integra ambas con `ConversationalRetrievalChain`.

## 1. Setup

In [ ]:
import warnings
from pathlib import Path

from langchain_classic.chains import ConversationalRetrievalChain
from langchain_classic.memory import ConversationBufferWindowMemory
from langchain_community.vectorstores import Chroma
from langchain_core._api.deprecation import suppress_langchain_deprecation_warning
from langchain_ollama import ChatOllama, OllamaEmbeddings

warnings.filterwarnings("ignore")

BASE_DIR = Path("../../").resolve()
VECTOR_DIR = str(BASE_DIR / "data" / "vector_db")

embeddings = OllamaEmbeddings(model="nomic-embed-text:latest")
vectorstore = Chroma(
    persist_directory=VECTOR_DIR,
    embedding_function=embeddings,
    collection_name="ml_papers",
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
llm = ChatOllama(model="llama3.2:1b", temperature=0)

print(f"✅ Chroma cargado desde {VECTOR_DIR}")

## 2. Cadena conversacional con RAG

`ConversationalRetrievalChain` hace dos cosas en cada turno:
1. Usa el **historial** para reformular la pregunta si hace falta.
2. **Recupera** chunks relevantes y genera la respuesta.

In [ ]:
memory = ConversationBufferWindowMemory(
    k=4,
    memory_key="chat_history",
    return_messages=True,
    output_key="answer",
)

with suppress_langchain_deprecation_warning():
    qa = ConversationalRetrievalChain.from_llm(
        llm=llm,
        retriever=retriever,
        memory=memory,
        return_source_documents=True,
    )

print("✅ Cadena RAG + memoria lista")

## 3. Diálogo de prueba

Observa cómo la **memoria** resuelve preguntas sobre el usuario y el **RAG** responde sobre los papers.

In [ ]:
def turn(question: str):
    with suppress_langchain_deprecation_warning():
        out = qa.invoke({"question": question})
    print(f"\n❓ {question}")
    print(f"💬 {out['answer']}")
    if out.get("source_documents"):
        src = out["source_documents"][0].metadata.get("source", "?")
        print(f"📄 Fuente: {Path(src).name}")

turn("Hola, soy Ana y estudio ingeniería de IA.")
turn("¿Recuerdas cómo me llamo y qué estudio?")
turn("¿Cuál es el proceso de ingestión de datos en FinGPT?")

## 4. Inspeccionar la memoria

In [ ]:
history = memory.load_memory_variables({})["chat_history"]
print(f"Turnos en memoria: {len(history)} mensajes\n")
for msg in history:
    role = "Usuario" if msg.type == "human" else "Asistente"
    print(f"{role}: {msg.content[:120]}..." if len(msg.content) > 120 else f"{role}: {msg.content}")

## 5. Resumen

- **Memoria** → coherencia del diálogo (pronombres, datos del usuario).
- **RAG** → hechos de documentos externos (papers, manuales, KB).
- En producción: `apps/rag_app/app.py` ya guarda mensajes en sesión; añadir memoria persistente con LangGraph Store para multi-sesión.

Siguiente: `04_advanced_rag.ipynb` — técnicas para mejorar el retrieval.